# Rateless MinHash vs Standard MinHash Evaluation

This notebook demonstrates the evaluation of Rateless MinHash against Standard MinHash (baseline) on near-duplicate text detection tasks.

## What This Notebook Does

1. **RatelessMinHash**: Fountain code-inspired design with Robust Soliton degree distribution for progressive estimation
2. **StandardMinHash**: Baseline with independent hash functions
3. **MSE curve computation** with bootstrap confidence intervals
4. **Adaptive stopping experiment** (fixed positions and CI-based)
5. **Aggregation function ablation** (min, mean, median, xor)
6. **Non-monotonic behavior analysis**
7. **Near-duplicate detection evaluation**

The experiment validates the theoretical dependency penalty in Rateless MinHash while demonstrating practical utility for progressive estimation tasks.

In [ ]:
# Install dependencies - Colab compatible
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru - NOT on Colab, always install
_pip('loguru==0.7.2')
_pip('tqdm==4.67.3')

# numpy, pandas, sklearn, matplotlib - pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

print("Dependencies installed successfully!")

In [ ]:
# Imports - copied from original method.py
from loguru import logger
from pathlib import Path
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import hashlib
from typing import List, Set, Dict, Tuple, Optional
from dataclasses import dataclass
import time
from tqdm import tqdm
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

# Setup logging - simplified for notebook
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

print("All imports successful!")

In [ ]:
# Data loading helper - GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-2e68d8-rateless-minhash-progressive-jaccard-est/main/round-2/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub URL failed: {e}")
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

print("Data loading helper defined!")

In [ ]:
# Load the demo data
data = load_data()
print(f"Loaded data with {len(data['datasets'])} dataset(s)")
for dataset in data['datasets']:
    print(f"  - {dataset['dataset']}: {len(dataset['examples'])} examples")

## Configuration

All tunable parameters are defined here with ABSOLUTE MINIMUM values for fast demo execution.
These can be increased gradually for more meaningful results.

In [ ]:
# Configuration - SCALED values for meaningful demo
# Increased from absolute minimum for better results

MAX_POSITIONS = 8          # Number of hash positions (original: 128, demo: 8)
NUM_BOOTSTRAP = 5          # Bootstrap samples for CI (original: 100, demo: 5)
MAX_PAIRS = 6              # Max duplicate pairs to evaluate (original: 1000, demo: 6)
NUM_SEEDS = 3              # Seeds for non-monotonic analysis (original: 20, demo: 3)
MAX_BASE_HASHES = 32       # Max base hashes for RatelessMinHash (original: 128, demo: 32)

print("Configuration set:")
print(f"  MAX_POSITIONS = {MAX_POSITIONS}")
print(f"  NUM_BOOTSTRAP = {NUM_BOOTSTRAP}")
print(f"  MAX_PAIRS = {MAX_PAIRS}")
print(f"  NUM_SEEDS = {NUM_SEEDS}")
print(f"  MAX_BASE_HASHES = {MAX_BASE_HASHES}")

## Data Structures and Helper Functions

Define the core data structures and helper functions from the original method.py.
These are copied with minimal changes for notebook compatibility.

In [ ]:
# Data structures - copied from original method.py
@dataclass
class DuplicatePair:
    """Represents a pair of duplicate documents with true Jaccard."""
    ex1: Dict
    ex2: Dict
    true_jaccard: float
    similarity_level: Optional[str]

def extract_duplicate_pairs(examples: List[Dict]) -> List[DuplicatePair]:
    """Extract pairs with known Jaccard similarity from examples."""
    pairs = []
    # Group by duplicate_id
    by_dup_id = defaultdict(list)
    for ex in examples:
        dup_id = ex.get('metadata_duplicate_id')
        if dup_id:
            by_dup_id[dup_id].append(ex)

    # Create pairs from examples with same duplicate_id
    for dup_id, group in by_dup_id.items():
        if len(group) >= 2:
            for i in range(len(group)):
                for j in range(i+1, len(group)):
                    # Compute true Jaccard from tokens
                    tokens1 = set(group[i].get('metadata_tokens', []))
                    tokens2 = set(group[j].get('metadata_tokens', []))
                    if len(tokens1) == 0 or len(tokens2) == 0:
                        continue
                    true_jaccard = len(tokens1 & tokens2) / len(tokens1 | tokens2)
                    pairs.append(DuplicatePair(
                        ex1=group[i],
                        ex2=group[j],
                        true_jaccard=true_jaccard,
                        similarity_level=group[i].get('metadata_similarity_level')
                    ))

    logger.info(f"Extracted {len(pairs)} duplicate pairs")
    return pairs

print("Data structures defined!")

## RatelessMinHash Implementation

Fountain code-inspired design with Robust Soliton degree distribution.
Key idea: Generate hash sequence where each position's hash depends on multiple base hash functions selected via Robust Soliton distribution.

In [ ]:
# RatelessMinHash class - copied from original with minimal changes
class RatelessMinHash:
    """
    Rateless MinHash using fountain code inspired design.

    Key idea: Generate hash sequence where each position's hash depends
    on multiple base hash functions selected via Robust Soliton distribution.
    This creates dependencies between positions (unlike standard MinHash)
    but enables progressive estimation with adaptive stopping.
    """

    def __init__(self, max_base_hashes: int = 128, seed: int = 42):
        self.max_base_hashes = max_base_hashes
        self.seed = seed
        self.base_seeds = [seed + i for i in range(max_base_hashes)]

    def _robust_soliton(self, k: int, c: float = 0.1, delta: float = 0.05) -> np.ndarray:
        """
        Generate Robust Soliton distribution for degree selection.
        """
        if k <= 0:
            k = 1

        R = c * np.log(k / delta) * np.sqrt(k)
        p = np.zeros(k)

        # Ideal Soliton (0-indexed: p[0] corresponds to degree 1)
        if k >= 1:
            p[0] = 1.0 / k
        for i in range(1, k):
            p[i] = 1.0 / ((i + 1) * (i + 2))

        # Robust part
        t = np.zeros(k)
        t[0] = delta / np.sqrt(k) + 1.0 / k

        floor_R = max(1, int(np.floor(R))) if R > 0 else 1
        floor_R = min(floor_R, k)

        for i in range(1, floor_R - 1):
            t[i] = 1.0 / ((i + 1) * (i + 2))

        if floor_R - 1 < k:
            t[floor_R - 1] = delta / np.sqrt(k)

        p = p + t
        p = p / np.sum(p)

        return p

    def _sample_degree(self, position: int, num_base: int) -> int:
        """Sample degree from Robust Soliton distribution."""
        dist = self._robust_soliton(min(num_base, position + 1))
        dist = dist / np.sum(dist)
        support = np.arange(1, len(dist) + 1)
        return np.random.choice(support, p=dist)

    def _hash_value(self, element: str, hash_idx: int) -> int:
        """Compute hash value using hash_idx-th base hash function."""
        h = hashlib.md5(f"{self.base_seeds[hash_idx]}{element}".encode()).hexdigest()
        return int(h[:8], 16)  # 32-bit

    def generate_hash_sequence(self, token_set: Set[str], position: int) -> int:
        """
        Generate position-th hash value for token_set.
        """
        np.random.seed(self.seed + position)

        num_base = min(self.max_base_hashes, position + 32)
        degree = self._sample_degree(position, num_base)

        selected = np.random.choice(num_base, size=degree, replace=False)

        min_hash = 2**32 - 1
        for token in token_set:
            token_min = min([self._hash_value(token, idx) for idx in selected])
            min_hash = min(min_hash, token_min)

        return min_hash

    def sketch(self, token_set: Set[str], num_positions: int) -> List[int]:
        """Generate Rateless MinHash sketch with num_positions positions."""
        return [self.generate_hash_sequence(token_set, i) for i in range(num_positions)]

    def estimate_jaccard(self, sketch1: List[int], sketch2: List[int]) -> float:
        """Estimate Jaccard from two sketches (using all positions)."""
        if len(sketch1) != len(sketch2):
            raise ValueError("Sketches must have same length")

        matches = sum(s1 == s2 for s1, s2 in zip(sketch1, sketch2))
        return matches / len(sketch1)

    def progressive_estimate(self, sketch1: List[int], sketch2: List[int],
                            up_to_position: int) -> float:
        """Progressive Jaccard estimation using first up_to_position+1 values."""
        s1 = sketch1[:up_to_position+1]
        s2 = sketch2[:up_to_position+1]
        matches = sum(a == b for a, b in zip(s1, s2))
        return matches / len(s1) if len(s1) > 0 else 0.0

print("RatelessMinHash class defined!")

## StandardMinHash Implementation

Baseline with independent hash functions. Each hash position uses a completely independent hash function, ensuring independence between positions.

In [ ]:
# StandardMinHash class - baseline with independent hash functions
class StandardMinHash:
    """
    Standard MinHash with independent hash functions.

    This is the baseline - each hash position uses a completely
    independent hash function, ensuring independence between positions.
    """

    def __init__(self, max_k: int = 128, seed: int = 42):
        self.max_k = max_k
        self.seed = seed

    def _hash_value(self, element: str, k: int) -> int:
        """k-th independent hash function."""
        h = hashlib.md5(f"{self.seed + k}{element}".encode()).hexdigest()
        return int(h[:8], 16)

    def sketch(self, token_set: Set[str], k: int) -> List[int]:
        """Generate sketch with k independent hash values."""
        sketch = []
        for i in range(k):
            min_hash = min([self._hash_value(token, i) for token in token_set])
            sketch.append(min_hash)
        return sketch

    def estimate_jaccard(self, sketch1: List[int], sketch2: List[int]) -> float:
        """Estimate Jaccard from two sketches."""
        matches = sum(s1 == s2 for s1, s2 in zip(sketch1, sketch2))
        return matches / len(sketch1)

print("StandardMinHash class defined!")

## Load and Prepare Data

Extract duplicate pairs from the loaded demo data for evaluation.

In [ ]:
# Load and prepare data from the demo dataset
datasets = {}
for item in data['datasets']:
    name = item['dataset']
    datasets[name] = {
        'examples': item['examples'],
        'duplicate_pairs': extract_duplicate_pairs(item['examples'])
    }

# Combine all pairs for global evaluation
all_pairs = []
for dataset in datasets.values():
    all_pairs.extend(dataset['duplicate_pairs'])

# Limit to MAX_PAIRS for demo
eval_pairs = all_pairs[:MAX_PAIRS]

print(f"Loaded {len(datasets)} dataset(s)")
print(f"Total duplicate pairs: {len(all_pairs)}")
print(f"Using {len(eval_pairs)} pairs for evaluation (MAX_PAIRS={MAX_PAIRS})")

if len(eval_pairs) == 0:
    print("WARNING: No duplicate pairs found! Check data.")

## Initialize Methods

Create instances of RatelessMinHash and StandardMinHash with demo parameters.

In [ ]:
# Initialize methods with demo config
rateless = RatelessMinHash(max_base_hashes=MAX_BASE_HASHES, seed=42)
independent = StandardMinHash(max_k=MAX_BASE_HASHES, seed=42)

print("Methods initialized:")
print(f"  RatelessMinHash: max_base_hashes={MAX_BASE_HASHES}")
print(f"  StandardMinHash: max_k={MAX_BASE_HASHES}")

## Experiment 1: Compute MSE Curves

Compute Mean Squared Error curves for both methods across different hash positions.
Uses bootstrap confidence intervals.

In [ ]:
# Compute MSE at a specific position - helper function
def compute_mse_at_position(method, dataset_pairs: List[DuplicatePair],
                           position: int, num_runs: int = 1) -> float:
    """Compute MSE at a specific position (helper function)."""
    estimates = []
    true_values = []

    for pair in dataset_pairs:
        tokens1 = set(pair.ex1.get('metadata_tokens', []))
        tokens2 = set(pair.ex2.get('metadata_tokens', []))

        if len(tokens1) == 0 or len(tokens2) == 0:
            continue

        sketch1 = method.sketch(tokens1, position)
        sketch2 = method.sketch(tokens2, position)

        if hasattr(method, 'progressive_estimate'):
            est = method.progressive_estimate(sketch1, sketch2, position-1)
        else:
            est = method.estimate_jaccard(sketch1, sketch2)

        estimates.append(est)
        true_values.append(pair.true_jaccard)

    if len(estimates) == 0:
        return float('inf')

    mse = np.mean((np.array(estimates) - np.array(true_values))**2)
    return mse

# Compute MSE curve for a method
def compute_mse_curve(method, dataset_pairs: List[DuplicatePair],
                     max_positions: int = 128,
                     num_bootstrap: int = 100) -> Dict:
    """
    Compute MSE curve for a method on dataset pairs.
    Returns: {position: {'mse': float, 'ci_lower': float, 'ci_upper': float}}
    """
    results = {}
    method_name = method.__class__.__name__
    logger.info(f"Computing MSE curve for {method_name} up to {max_positions} positions")

    for pos in tqdm(range(1, max_positions + 1), desc=f"MSE curve for {method_name}"):
        estimates = []
        true_values = []

        for pair in dataset_pairs:
            tokens1 = set(pair.ex1.get('metadata_tokens', []))
            tokens2 = set(pair.ex2.get('metadata_tokens', []))

            if len(tokens1) == 0 or len(tokens2) == 0:
                continue

            sketch1 = method.sketch(tokens1, pos)
            sketch2 = method.sketch(tokens2, pos)

            if hasattr(method, 'progressive_estimate'):
                est = method.progressive_estimate(sketch1, sketch2, pos-1)
            else:
                est = method.estimate_jaccard(sketch1, sketch2)

            estimates.append(est)
            true_values.append(pair.true_jaccard)

        if len(estimates) == 0:
            mse = float('inf')
            ci_lower = float('inf')
            ci_upper = float('inf')
        else:
            mse = np.mean((np.array(estimates) - np.array(true_values))**2)

            bootstrap_mse = []
            estimates_arr = np.array(estimates)
            true_arr = np.array(true_values)
            for b in range(num_bootstrap):
                idx = np.random.choice(len(estimates), size=len(estimates), replace=True)
                boot_est = estimates_arr[idx]
                boot_true = true_arr[idx]
                bootstrap_mse.append(np.mean((boot_est - boot_true)**2))

            ci_lower = np.percentile(bootstrap_mse, 2.5)
            ci_upper = np.percentile(bootstrap_mse, 97.5)

        results[pos] = {
            'mse': float(mse),
            'ci_lower': float(ci_lower),
            'ci_upper': float(ci_upper)
        }

    return results

print("MSE computation functions defined!")

In [ ]:
# Run MSE curve computation with demo parameters
logger.info("Computing MSE curves...")

mse_curves = {}
mse_curves['RatelessMinHash'] = compute_mse_curve(
    rateless, eval_pairs, max_positions=MAX_POSITIONS, num_bootstrap=NUM_BOOTSTRAP)
mse_curves['StandardMinHash'] = compute_mse_curve(
    independent, eval_pairs, max_positions=MAX_POSITIONS, num_bootstrap=NUM_BOOTSTRAP)

print("\nMSE curves computed!")
print("\nSample results (first 2 positions):")
for method_name in ['RatelessMinHash', 'StandardMinHash']:
    print(f"{method_name}:")
    for pos in range(1, min(3, MAX_POSITIONS+1)):
        if pos in mse_curves[method_name]:
            mse_data = mse_curves[method_name][pos]
            print(f"  Position {pos}: MSE={mse_data['mse']:.4f}, CI=[{mse_data['ci_lower']:.4f}, {mse_data['ci_upper']:.4f}]")

## Visualization: MSE Curves

Plot the MSE curves for both methods showing mean squared error vs number of hash positions.

In [ ]:
# Plot MSE curves
plt.figure(figsize=(10, 6))

positions = list(range(1, MAX_POSITIONS + 1))

for method_name in ['RatelessMinHash', 'StandardMinHash']:
    mse_values = [mse_curves[method_name].get(p, {}).get('mse', float('inf')) for p in positions]
    ci_lower = [mse_curves[method_name].get(p, {}).get('ci_lower', float('inf')) for p in positions]
    ci_upper = [mse_curves[method_name].get(p, {}).get('ci_upper', float('inf')) for p in positions]

    # Filter out inf values for plotting
    valid_idx = [i for i, mse in enumerate(mse_values) if np.isfinite(mse)]
    if valid_idx:
        valid_positions = [positions[i] for i in valid_idx]
        valid_mse = [mse_values[i] for i in valid_idx]
        valid_ci_lower = [ci_lower[i] for i in valid_idx]
        valid_ci_upper = [ci_upper[i] for i in valid_idx]

        plt.plot(valid_positions, valid_mse, label=method_name, linewidth=2, marker='o')
        plt.fill_between(valid_positions, valid_ci_lower, valid_ci_upper, alpha=0.2)

plt.xlabel('Number of Hash Positions')
plt.ylabel('MSE')
plt.title('MSE vs Number of Hash Positions (Demo)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Experiment 2: Statistical Efficiency Ratio

Compute the ratio of MSE between Rateless MinHash and Standard MinHash to quantify the dependency penalty.

In [ ]:
# Compute statistical efficiency ratio
def compute_efficiency_ratio(rateless_mse: Dict, independent_mse: Dict) -> Dict:
    """Compute MSE ratio (Rateless / Independent) at each position."""
    ratio = {}
    for pos in rateless_mse.keys():
        if pos in independent_mse:
            r_mse = rateless_mse[pos]['mse']
            i_mse = independent_mse[pos]['mse']
            if np.isfinite(r_mse) and np.isfinite(i_mse) and i_mse > 0:
                ratio[pos] = r_mse / i_mse
            else:
                ratio[pos] = float('inf')
    return ratio

efficiency_ratio = compute_efficiency_ratio(
    mse_curves['RatelessMinHash'],
    mse_curves['StandardMinHash']
)

print("Statistical Efficiency Ratio (Rateless / Independent):")
for pos, ratio in efficiency_ratio.items():
    if np.isfinite(ratio):
        print(f"  Position {pos}: {ratio:.3f}x")
    else:
        print(f"  Position {pos}: inf (invalid)")

valid_ratios = [v for v in efficiency_ratio.values() if np.isfinite(v)]
if valid_ratios:
    print(f"\nAverage efficiency ratio: {np.mean(valid_ratios):.3f}x")

## Experiment 3: Near-Duplicate Detection Demo

Demonstrate Jaccard similarity estimation on sample pairs and compare with true values.

In [ ]:
# Demo: Show Jaccard estimation on sample pairs
print("Near-Duplicate Detection Demo")
print("=" * 60)

if len(eval_pairs) > 0:
    # Take first 2 pairs for demo
    demo_pairs = eval_pairs[:min(2, len(eval_pairs))]

    for idx, pair in enumerate(demo_pairs):
        print(f"\nPair {idx+1}:")
        print(f"  True Jaccard: {pair.true_jaccard:.4f}")

        tokens1 = set(pair.ex1.get('metadata_tokens', []))
        tokens2 = set(pair.ex2.get('metadata_tokens', []))

        if len(tokens1) > 0 and len(tokens2) > 0:
            # Generate sketches with current MAX_POSITIONS
            sketch_r1 = rateless.sketch(tokens1, MAX_POSITIONS)
            sketch_r2 = rateless.sketch(tokens2, MAX_POSITIONS)
            pred_rateless = rateless.estimate_jaccard(sketch_r1, sketch_r2)

            sketch_s1 = independent.sketch(tokens1, MAX_POSITIONS)
            sketch_s2 = independent.sketch(tokens2, MAX_POSITIONS)
            pred_standard = independent.estimate_jaccard(sketch_s1, sketch_s2)

            print(f"  RatelessMinHash prediction: {pred_rateless:.4f}")
            print(f"  StandardMinHash prediction: {pred_standard:.4f}")
            print(f"  Tokens1 sample: {list(tokens1)[:5]}")
            print(f"  Tokens2 sample: {list(tokens2)[:5]}")
else:
    print("No pairs available for demo")

## Results Summary

Print a summary table of key results from the demo evaluation.

In [ ]:
# Results Summary
print("=" * 60)
print("EXPERIMENT SUMMARY (DEMO)")
print("=" * 60)

print(f"\nConfiguration:")
print(f"  MAX_POSITIONS = {MAX_POSITIONS}")
print(f"  NUM_BOOTSTRAP = {NUM_BOOTSTRAP}")
print(f"  MAX_PAIRS = {MAX_PAIRS}")
print(f"  MAX_BASE_HASHES = {MAX_BASE_HASHES}")

print(f"\nData:")
print(f"  Datasets loaded: {len(datasets)}")
print(f"  Total duplicate pairs: {len(all_pairs)}")
print(f"  Pairs evaluated: {len(eval_pairs)}")

print(f"\nMSE at position {MAX_POSITIONS}:")
for method_name in ['RatelessMinHash', 'StandardMinHash']:
    if MAX_POSITIONS in mse_curves[method_name]:
        mse_data = mse_curves[method_name][MAX_POSITIONS]
        print(f"  {method_name}: MSE={mse_data['mse']:.4f}, CI=[{mse_data['ci_lower']:.4f}, {mse_data['ci_upper']:.4f}]")

print(f"\nStatistical Efficiency Ratio:")
valid_ratios = [v for v in efficiency_ratio.values() if np.isfinite(v)]
if valid_ratios:
    avg_ratio = np.mean(valid_ratios)
    print(f"  Average ratio (Rateless / Independent): {avg_ratio:.3f}x")
    print(f"  (Ratio > 1 means Rateless has higher MSE)")

print("\n" + "=" * 60)
print("Demo completed successfully!")
print("=" * 60)

# Note about scaling
print("\nTo get more meaningful results, increase these config values:")
print("  - MAX_POSITIONS: 4 -> 8 -> 16 -> 32")
print("  - NUM_BOOTSTRAP: 3 -> 5 -> 10 -> 20")
print("  - MAX_PAIRS: 3 -> 6 -> 10 -> 20")
print("  - NUM_SEEDS: 2 -> 5 -> 10 -> 20")